# IB Async Reference Guide

This notebook serves as a reference for using `ib_async` to interact with Interactive Brokers. It covers connecting, defining contracts, placing various types of orders, canceling orders, and basic logic.

In [ ]:
from ib_async import *
import asyncio

# Start the event loop. This is required for ib_async to work in a Jupyter Notebook.
util.startLoop()

## 1. Connecting to IB

Ensure your TWS or IB Gateway is running and configured to accept API connections.
*   **TWS Live**: Port 7496
*   **TWS Paper**: Port 7497
*   **Gateway Live**: Port 4001
*   **Gateway Paper**: Port 4002

In [ ]:
ib = IB()

# Connect to TWS/Gateway
# Change port and clientId as needed. clientId=0 is usually for the main controller.
if not ib.isConnected():
    ib.connect('127.0.0.1', 7497, clientId=1)
    print("Connected!")
else:
    print("Already connected.")

## 2. Defining Contracts

Before doing anything, you need to define the financial instrument (Contract). `ib_async` provides helpers like `Stock`, `Option`, `Future`, `Forex`, etc.

In [ ]:
# Define a Stock Contract (e.g., Apple)
# Symbol: AAPL, Exchange: SMART, Currency: USD
contract = Stock('AAPL', 'SMART', 'USD')

# It is good practice to 'qualify' the contract. 
# This fetches the unique conId and other details from IB to ensure the contract is valid.
ib.qualifyContracts(contract)

print(f"Qualified Contract: {contract}")

## 3. Placing Orders

Here are examples of common order types. Note that `placeOrder` is non-blocking and returns a `Trade` object that updates in real-time.

### Market Order

In [ ]:
# Create a Market Order to BUY 1 share
market_order = MarketOrder('BUY', 1)

# Place the order
trade_market = ib.placeOrder(contract, market_order)

# View the trade status
print(f"Market Order Status: {trade_market.orderStatus.status}")

# You can wait for the order to fill (optional)
# ib.sleep(1) # Wait a bit for updates

### Limit Order

In [ ]:
# Create a Limit Order to BUY 1 share at a specific price (e.g., $100.00)
# This will likely not fill immediately if the price is below market.
limit_price = 100.00
limit_order = LimitOrder('BUY', 1, limit_price)

trade_limit = ib.placeOrder(contract, limit_order)

print(f"Limit Order Placed: {trade_limit.order}")
print(f"Status: {trade_limit.orderStatus.status}")

### Stop Order

In [ ]:
# Create a Stop Order to SELL 1 share if price drops to $150.00
stop_price = 150.00
stop_order = StopOrder('SELL', 1, stop_price)

trade_stop = ib.placeOrder(contract, stop_order)

print(f"Stop Order Placed: {trade_stop.order}")

## 4. Canceling Orders

You can cancel an order using the order object or the trade object.

In [ ]:
# Cancel the Limit Order we placed earlier
if trade_limit.isActive():
    ib.cancelOrder(limit_order)
    print("Canceling Limit Order...")
    
    # Wait a moment for the cancellation to propagate
    ib.sleep(1)
    print(f"New Status: {trade_limit.orderStatus.status}")
else:
    print("Order is not active (already filled or cancelled).")

## 5. Intro to Complex Logic

You can use `ib.reqMktData` to get real-time data and build logic around it. Since `ib_async` is asynchronous, you can run loops or use event callbacks.

In [ ]:
# Request real-time market data for the contract
ticker = ib.reqMktData(contract, '', False, False)

# Wait for data to arrive
ib.sleep(2)

print(f"Current Market Price for {contract.symbol}: {ticker.marketPrice()}")

# Simple Logic Example: Check price and print action
current_price = ticker.marketPrice()
target_price = 200.00

if current_price < target_price:
    print(f"Price {current_price} is below target {target_price}. Signal: BUY")
    # You could place an order here:
    # ib.placeOrder(contract, MarketOrder('BUY', 1))
else:
    print(f"Price {current_price} is above target {target_price}. Signal: WAIT")

In [ ]:
# Event-Driven Logic Example
# Define a function that runs whenever a ticker updates

def onTickerUpdate(t):
    if t.contract.symbol == 'AAPL':
        print(f"Update: Bid={t.bid}, Ask={t.ask}, Last={t.last}")
        if t.last and t.last > 250:
             print("Price spike detected!")

# Register the callback
ticker.updateEvent += onTickerUpdate

# Let it run for a few seconds to see updates
print("Listening for updates (5 seconds)...")
ib.sleep(5)

# Unregister the callback to stop printing
ticker.updateEvent -= onTickerUpdate
print("Stopped listening.")

In [ ]:
# Disconnect when done
ib.disconnect()